In [ ]:

import time
from flax import nnx
import jax.numpy as jnp


def make_time_rngs():
    seed = int(time.time()) 
    key = nnx.Rngs(seed)
    params_key, dropout_key = nnx.random.split(key)
    return nnx.Rngs(params=params_key, dropout=dropout_key)


class LSTMEncoder(nnx.Module):
    def __init__(self, input_size: int, hidden_size: list[int], rngs: nnx.Rngs):
        self.rngs = rngs
        self.layers = []
        in_size = input_size
        for h in hidden_size:
            layer = nnx.LSTMCell(in_features=in_size, hidden_features=h, rngs=rngs)
            self.layers.append(layer)
            in_size = h

    def __call__(self, inputs):
        batch_size, _, _ = x.shape
        for layer in self.layers:
            def step(carry, x_t):
                y, new_carry = layer(x_t, carry)
                return new_carry, y
            carry = nnx.LSTMCell.initialize_carry(
                self.rngs,
                batch_dims=(batch_size,),
                size=layer.hidden_features,
            )
            carry, outputs = nnx.scan(
                step,
                variable_axes={},
                split_rngs={"params": False},
                in_axes=1,
                out_axes=1,
            )(carry, x)
            x = outputs
        return x, carry


class LSTMDecoder(nnx.Module):
    def __init__(self, latent_size: int, output_size: int, hidden_size: list[int], seq_len: int, rngs: nnx.Rngs):
        self.rngs = rngs
        self.seq_len = seq_len
        dimension = [latent_size] + hidden_size
        self.layers = [
            nnx.LSTMCell(
                in_features=dimension[i], 
                hidden_features=dimension[i + 1], 
                rngs=rngs
            ) for i in range(len(hidden_size))
        ]
        self.output_layer = nnx.Linear(
            in_features=hidden_size[-1], 
            out_features=output_size, 
            rngs=rngs
        )

    def __call__(self, latent, encoder_carry):
        batch_size = latent.shape[0]
        x = jnp.tile(latent[:, None, :], (1, self.seq_len, 1))
        for layer_idx, layer in enumerate(self.layers):
            def step(carry, x_t):
                y, new_carry = layer(x_t, carry)
                return new_carry, y
            carry = encoder_carry if layer_idx == 0 else nnx.LSTMCell.initialize_carry(
                self.rngs,
                batch_dims=(batch_size,),
                size=layer.hidden_features,
            )
            carry, outputs = nnx.scan(
                step,
                variable_axes={},
                split_rngs={"params": False},
                in_axes=1,
                out_axes=1,
            )(carry, x)
            x = outputs
        reconstruction = self.output_layer(x)
        return reconstruction


class LSTMAutoencoder(nnx.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size_enc: list[int],
        hidden_size_dec: list[int],
        seq_len: int,
        rngs: nnx.Rngs,
    ):
        self.rngs = rngs
        self.seq_len = seq_len

        self.encoder = LSTMEncoder(
            input_size=input_size,
            hidden_size=hidden_size_enc,
            rngs=rngs,
        )

        self.decoder = LSTMDecoder(
            latent_size=hidden_size_enc[-1], 
            output_size=input_size,
            hidden_size=hidden_size_dec,
            seq_len=seq_len,
            rngs=rngs,
        )

    def __call__(self, x):
        latent_seq, encoder_carry = self.encoder(x)
        latent = encoder_carry[0]
        reconstruction = self.decoder(latent, encoder_carry)
        return reconstruction, latent


In [2]:
import pandas as pd

dataset = pd.read_csv('02-14-2018.csv')

dataset

# benign_data = dataset[dataset['Label'] == 'Benign'].copy()
# anomaly_data = dataset[dataset['Label'] != 'Benign'].copy()

# del dataset

# train_data = benign_data.iloc[:300000]
# test_benign_data = benign_data.iloc[300000:]

# del benign_data

# test_data = pd.concat([test_benign_data, anomaly_data], ignore_index=True)

# del test_benign_data, anomaly_data

# train_data.info()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,14/02/2018 08:31:01,112641719,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320859.5,139.300036,56320958,56320761,Benign
1,0,0,14/02/2018 08:33:50,112641466,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320733.0,114.551299,56320814,56320652,Benign
2,0,0,14/02/2018 08:36:39,112638623,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56319311.5,301.934596,56319525,56319098,Benign
3,22,6,14/02/2018 08:40:13,6453966,15,10,1239,2273,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
4,22,6,14/02/2018 08:40:23,8804066,14,11,1143,2209,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048570,80,6,14/02/2018 10:53:23,10156986,5,5,1089,1923,587,0,...,20,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
1048571,80,6,14/02/2018 10:53:33,117,2,0,0,0,0,0,...,20,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
1048572,80,6,14/02/2018 10:53:28,5095331,3,1,0,0,0,0,...,20,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
1048573,80,6,14/02/2018 10:53:28,5235511,3,1,0,0,0,0,...,20,0.0,0.0,0,0,0.0,0.000000,0,0,Benign


In [3]:
dataset["Label"].unique()

array(['Benign', 'FTP-BruteForce', 'SSH-Bruteforce'], dtype=object)

In [ ]:
import jax
import optax 

def mse_loss(model, batch):
    x = batch
    recon, _ = model(x)
    loss = jnp.mean((x - recon) ** 2)
    return loss, recon

@nnx.jit
def train_step(carry, batch):
    model, optimizer, metrics = carry
    (loss, recon), grads = nnx.value_and_grad(mse_loss, has_aux=True)(model, batch)
    optimizer.update(grads)
    metrics.update(loss=loss)
    return (model, optimizer, metrics), None

def pretrain(model, train_data, batch_size=128, epochs=10):
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate=1e-3))
    metrics = nnx.MultiMetric(loss=nnx.metrics.Average('loss'))
    
    n_batches = train_data.shape[0] // batch_size

    for epoch in range(epochs):
        idx = jax.random.permutation(nnx.Rngs(0), train_data.shape[0])
        x_shuffled = train_data[idx]
        
        for i in range(n_batches):
            batch = x_shuffled[i*batch_size:(i+1)*batch_size]
            carry = (model, optimizer, metrics)
            (model, optimizer, metrics), _ = nnx.scan(train_step)(carry, batch)
        
        epoch_loss = metrics.compute()['loss']
        metrics.reset()
        print(f"[Pretrain] Epoch {epoch} | Loss: {epoch_loss:.4f}")
    
    return model, optimizer
